# CKD — XAI extension

The reusable XAI framework is applied to the pre-specified Random Forest model with 200 trees and `random_state=42`. Imputation and categorical encoding are fitted using training data only.

The framework generates explanations and assesses retraining-based stability, using accuracy for scoring. Results are exported to `../../results/ckd`.

## Minimal model and data preparation

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

# =========================
# 1. LOAD + BASIC CLEANING
# =========================
file_path = "../../data/ckd/kidney_disease.csv"
DATASET_NAME = "kidney_disease"
df = pd.read_csv(file_path)

df = df.replace(r'^\s+|\s+$', '', regex=True)
df = df.replace('?', np.nan)
df = df.replace('\t?', np.nan)
df = df.replace({'\tyes': 'yes', '\tno': 'no', ' yes': 'yes', ' no': 'no'})

# find target
possible_targets = ['classification', 'class', 'target', 'label']
target_col = next((c for c in possible_targets if c in df.columns), None)
if target_col is None:
    raise ValueError("Target column not found.")

df[target_col] = df[target_col].astype(str).str.strip().str.lower()
df[target_col] = df[target_col].replace({
    'ckd\t': 'ckd',
    'notckd': 'notckd',
    'not ckd': 'notckd'
})

df = df[df[target_col].isin(['ckd', 'notckd'])].copy()

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X = X.drop(columns=['id'])

#  numeric conversion
for col in X.columns:
    converted = pd.to_numeric(X[col], errors='coerce')
    if converted.notna().sum() == X[col].notna().sum():
        X[col] = converted

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# target encode
y = y.map({'ckd': 1, 'notckd': 0})

print("Shape:", X.shape)
print("Target counts:\n", y.value_counts())
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

# =========================
# 2. SPLIT FIRST (70:30)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# =========================
# 3. IMPUTATION USING TRAIN ONLY
# =========================
rng = np.random.default_rng(42)

def random_sample_impute_train_test(train_s, test_s, rng_obj):
    train_s = train_s.copy()
    test_s = test_s.copy()

    observed = train_s.dropna().values
    if len(observed) == 0:
        return train_s, test_s

    # impute train from train observed values
    train_missing = train_s.isna()
    if train_missing.sum() > 0:
        sampled_train = rng_obj.choice(observed, size=train_missing.sum(), replace=True)
        train_s.loc[train_missing] = sampled_train

    # recompute observed after train fill
    observed = train_s.dropna().values

    # impute test from train distribution
    test_missing = test_s.isna()
    if test_missing.sum() > 0:
        sampled_test = rng_obj.choice(observed, size=test_missing.sum(), replace=True)
        test_s.loc[test_missing] = sampled_test

    return train_s, test_s

# numeric -> random sample, fallback mean
for col in numeric_cols:
    try:
        X_train[col], X_test[col] = random_sample_impute_train_test(X_train[col], X_test[col], rng)
    except Exception:
        mean_val = X_train[col].mean()
        X_train[col] = X_train[col].fillna(mean_val)
        X_test[col] = X_test[col].fillna(mean_val)

# categorical -> random sample, fallback mode
for col in categorical_cols:
    try:
        X_train[col], X_test[col] = random_sample_impute_train_test(X_train[col], X_test[col], rng)
    except Exception:
        mode_val = X_train[col].mode(dropna=True)
        if len(mode_val) > 0:
            fill_val = mode_val.iloc[0]
            X_train[col] = X_train[col].fillna(fill_val)
            X_test[col] = X_test[col].fillna(fill_val)

# final fallback
for col in numeric_cols:
    mean_val = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

for col in categorical_cols:
    mode_val = X_train[col].mode(dropna=True)
    if len(mode_val) > 0:
        fill_val = mode_val.iloc[0]
        X_train[col] = X_train[col].fillna(fill_val)
        X_test[col] = X_test[col].fillna(fill_val)

# =========================
# 4. CATEGORICAL ENCODING USING TRAIN ONLY
# =========================

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols].astype(str)
)

X_test[categorical_cols] = encoder.transform(
    X_test[categorical_cols].astype(str)
)



Shape: (400, 24)
Target counts:
 classification
1    250
0    150
Name: count, dtype: int64
Numeric: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
Categorical: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [2]:
# ==========================================
# SAVE NO-LEAKAGE OBJECTS
# ==========================================

X_train_no_leakage = X_train.copy()
X_test_no_leakage = X_test.copy()

y_train_no_leakage = y_train.copy()
y_test_no_leakage = y_test.copy()

models_no_leakage = {
    "RF": RandomForestClassifier(n_estimators=200, random_state=42)
}


In [3]:
# ==========================================
# SELECT FINAL CKD MODEL
# ==========================================

best_model_name = "RF"
print("Pre-specified XAI model from source study:", best_model_name)

from sklearn.base import clone

best_model = clone(
    models_no_leakage[best_model_name]
)

best_model.fit(
    X_train_no_leakage,
    y_train_no_leakage
)

# ==========================================
# STABILITY
# ==========================================

def train_model_fn(seed):

    model = clone(
        models_no_leakage[best_model_name]
    )

    params = model.get_params(deep=False)

    if "random_state" in params:
        model.set_params(random_state=seed)

    model.fit(
        X_train_no_leakage,
        y_train_no_leakage
    )

    return model




Pre-specified XAI model from source study: RF


## Call the reusable XAI pipeline

In [4]:
# ==========================================
# IMPORT XAI FRAMEWORK
# ==========================================

import sys
from pathlib import Path

FRAMEWORK_DIR = Path.cwd().parent / "xai_framework"

if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline




c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
config = {
    "dataset_name": "ckd",

    "model": best_model,

    "X_train": X_train_no_leakage,
    "X_test": X_test_no_leakage,

    "y_train": y_train_no_leakage,
    "y_test": y_test_no_leakage,

    "feature_names": list(
        X_train_no_leakage.columns
    ),

    "train_model_fn": train_model_fn,

    "scoring": "accuracy",

    "output_dir": "../../results/ckd"
}

experiment = run_pipeline(config)

Results exported to: ../../results/ckd


## XAI framework output sanity checks

In [6]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: ckd
Model: RandomForestClassifier
X_train shape: (280, 24)
X_test shape: (120, 24)
y_train length: 280
y_test length: 120
Feature names: 24


#### Consistency

In [7]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


#### Fitted model predicts on the configured data

In [8]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.0 1.0
Predictions: (array([0, 1]), array([45, 75]))
Prediction test: OK


#### Results from Framework

In [9]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime'])
dict_keys(['raw', 'summary', 'rank_stability'])


#### Global XAI

In [10]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,Method
11,sc,0.172794,tree_importance
2,sg,0.172209,tree_importance
14,hemo,0.151966,tree_importance
15,pcv,0.137196,tree_importance
3,al,0.073456,tree_importance


Permutation:


,Feature,Importance,Std
2,sg,0.124583,0.024363
11,sc,0.008750,0.006706
15,pcv,0.005833,0.012388
17,rc,0.002500,0.003819
3,al,0.002500,0.005336


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,sg,0.093746,-0.004480
1,sc,0.082862,-0.015533
2,hemo,0.078469,-0.003660
3,pcv,0.074007,-0.001781
4,al,0.043660,-0.000041


#### Feature coverage

In [11]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 24
Permutation features: 24
SHAP features: 24
Expected features: 24


#### Broken values

In [12]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


#### Rank agreement

In [13]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.9808695652173912), 'Native_vs_Permutation': np.float64(0.6407842490812532), 'SHAP_vs_Permutation': np.float64(0.6417551343071339)}


,Feature,Importance_x,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,sc,0.172794,tree_importance,1.0,0.082862,-0.015533,2.0,0.008750,0.006706,2.0
1,sg,0.172209,tree_importance,2.0,0.093746,-0.004480,1.0,0.124583,0.024363,1.0
2,hemo,0.151966,tree_importance,3.0,0.078469,-0.003660,3.0,-0.009167,0.006401,24.0
3,pcv,0.137196,tree_importance,4.0,0.074007,-0.001781,4.0,0.005833,0.012388,3.0
4,al,0.073456,tree_importance,5.0,0.043660,-0.000041,5.0,0.002500,0.005336,5.0


#### Representative cases

In [14]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
0,TP,1,1,1.00
1,TN,0,0,0.00
53,FP,0,1,0.69
110,FN,1,0,0.40


#### LIME

In [15]:
lime_df = local_results["lime"]

print("LIME rows:", len(lime_df))
display(lime_df.head())

LIME rows: 40


,CaseIndex,Prediction,Probability,Rule,Weight
0,0,1,1.0,sc > 3.20,0.068737
1,0,1,1.0,10.30 < hemo <= 12.70,0.068229
2,0,1,1.0,0.00 < htn <= 1.00,0.064272
3,0,1,1.0,32.00 < pcv <= 41.00,0.047634
4,0,1,1.0,al > 2.00,0.046838


#### Stability

In [16]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Seed,Rank
0,age,0.0000,0.000000,1,11.5
1,bp,0.0000,0.000000,1,11.5
2,sg,0.1050,0.015899,1,1.0
3,al,-0.0025,0.003819,1,23.0
4,su,0.0000,0.000000,1,11.5


   Feature  MeanImportance  StdImportance  MeanRank    StdRank
20      sg        0.104167       0.029486       1.0   0.000000
19      sc        0.005667       0.003354       2.5   0.707107
14     pcv        0.002000       0.006887      15.6  10.691118
17     rbc        0.000500       0.001118      10.1   3.927467
18      rc        0.000333       0.000745      10.4   3.305299
11     htn        0.000333       0.001728      11.4   9.838699
0      age        0.000000       0.000000      12.1   1.557241
2      ane        0.000000       0.000000      12.1   1.557241
16     pot        0.000000       0.000000      12.1   1.557241
21     sod        0.000000       0.000000      12.1   1.557241
15      pe        0.000000       0.000000      12.1   1.557241
13     pcc        0.000000       0.000000      12.1   1.557241
4       ba        0.000000       0.000000      12.1   1.557241
6       bp        0.000000       0.000000      12.1   1.557241
12      pc        0.000000       0.000000      12.1   1

#### Exported folder

In [17]:
from pathlib import Path

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\ckd
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
lime_explanations.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [18]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,cad,appet,pe,ane,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,71.0,60.0,1.015,4.0,0.0,1.0,1.0,0.0,0.0,118.0,...,0.0,1.0,1.0,0.0,1,1,1.00,0.50,TP,0
1,34.0,70.0,1.025,0.0,0.0,1.0,1.0,0.0,0.0,87.0,...,0.0,0.0,0.0,0.0,0,0,0.00,0.50,TN,1
53,40.0,80.0,1.020,0.0,0.0,1.0,1.0,0.0,0.0,119.0,...,0.0,0.0,0.0,0.0,0,1,0.69,0.19,FP,53
110,34.0,60.0,1.020,0.0,0.0,1.0,1.0,0.0,0.0,117.0,...,0.0,0.0,1.0,0.0,1,0,0.40,0.10,FN,110


In [19]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("CKD CaseIndex exists in memory.")

CKD CaseIndex exists in memory.


In [20]:
ckd_cases = pd.read_csv(
    Path("../../results/ckd/representative_cases.csv")
)

print(ckd_cases.columns.tolist())
display(ckd_cases)

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,cad,appet,pe,ane,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,71.0,60.0,1.015,4.0,0.0,1.0,1.0,0.0,0.0,118.0,...,0.0,1.0,1.0,0.0,1,1,1.00,0.50,TP,0
1,34.0,70.0,1.025,0.0,0.0,1.0,1.0,0.0,0.0,87.0,...,0.0,0.0,0.0,0.0,0,0,0.00,0.50,TN,1
2,40.0,80.0,1.020,0.0,0.0,1.0,1.0,0.0,0.0,119.0,...,0.0,0.0,0.0,0.0,0,1,0.69,0.19,FP,53
3,34.0,60.0,1.020,0.0,0.0,1.0,1.0,0.0,0.0,117.0,...,0.0,0.0,1.0,0.0,1,0,0.40,0.10,FN,110
